# Human Resources Candidate Ranking
## Notebook 03 — Final Stage-1 Ranking and Stage-2 Relevance Feedback

Selected Stage-1 fit and recruiter-feedback reranking; run from the repository root.

In [1]:
import hashlib
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

LABELLED_DATA = "potential-talents-labelled.csv"
QUERIES = ("aspiring human resources", "seeking human resources")
SELECTED_ALPHA = 10.0
SELECTED_FEATURES = [
    "bm25", "tfidf_word", "tfidf_char",
    "jaccard", "containment", "hr_overlap", "intent_overlap"
]
HR_TERMS = {"human", "resource", "resources", "hr", "hris"}
INTENT_TERMS = {"aspiring", "seeking", "student", "internship", "entry", "entrylevel"}

def surface_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_text(text):
    text = surface_text(text)
    text = re.sub(r"\bhr\b", " human resources ", text)
    text = re.sub(r"\bentry level\b", " entrylevel ", text)
    return re.sub(r"\s+", " ", text).strip()

def parse_connections(value):
    match = re.search(r"\d+", str(value).replace(",", ""))
    return float(match.group()) if match else np.nan

def stable_group_key(job_title, location, connection_num):
    payload = "|".join([
        normalize_text(job_title),
        normalize_text(location),
        "" if pd.isna(connection_num) else f"{float(connection_num):g}",
    ])
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

df = pd.read_csv(LABELLED_DATA)
assert len(df) == 104
assert df["id"].is_unique
assert df["final_revised_grade"].notna().all()

df["connection_num"] = df["connection"].map(parse_connections)
df["title_norm"] = df["job_title"].map(normalize_text)
df["group_key"] = [
    stable_group_key(t, l, c)
    for t, l, c in zip(df["job_title"], df["location"], df["connection_num"])
]
assert df.groupby("group_key")["final_revised_grade"].nunique().max() == 1

groups = (
    df.sort_values("id")
      .groupby("group_key", sort=False)
      .agg(
          canonical_id=("id", "min"),
          member_ids=("id", lambda s: tuple(int(x) for x in s)),
          duplicate_count=("id", "size"),
          job_title=("job_title", "first"),
          location=("location", "first"),
          connection_num=("connection_num", "first"),
          title_norm=("title_norm", "first"),
          relevance=("final_revised_grade", "first"),
      )
      .reset_index()
      .sort_values("canonical_id")
      .reset_index(drop=True)
)
assert len(groups) == 53

display(pd.DataFrame({
    "quantity": ["raw rows", "unique profile groups", "selected alpha", "connection feature used"],
    "value": [len(df), len(groups), SELECTED_ALPHA, False],
}))

def minmax(values):
    values = np.asarray(values, dtype=float)
    lo, hi = np.nanmin(values), np.nanmax(values)
    return (values - lo) / (hi - lo) if hi > lo else np.zeros_like(values)

def bm25_statistics(documents):
    tokenized = [doc.split() for doc in documents]
    n_docs = len(tokenized)
    avgdl = float(np.mean([len(tokens) for tokens in tokenized]))
    doc_freq = Counter()
    for tokens in tokenized:
        doc_freq.update(set(tokens))
    return n_docs, avgdl, doc_freq

def bm25_score_documents(documents, query, n_docs, avgdl, doc_freq, k1=1.5, b=0.75):
    query_tokens = query.split()
    scores = []
    for document in documents:
        tokens = document.split()
        tf = Counter(tokens)
        dl = len(tokens)
        score = 0.0
        for term in query_tokens:
            df = doc_freq.get(term, 0)
            idf = math.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
            f = tf.get(term, 0)
            if f:
                norm = 1 - b + b * dl / max(avgdl, 1e-12)
                score += idf * (f * (k1 + 1)) / (f + k1 * norm)
        scores.append(score)
    return np.asarray(scores, dtype=float)

def build_full_features(profile_groups):
    documents = profile_groups["title_norm"].tolist()
    queries = [normalize_text(q) for q in QUERIES]

    word_vectorizer = TfidfVectorizer(
        tokenizer=str.split, preprocessor=None, token_pattern=None,
        ngram_range=(1, 2), sublinear_tf=True
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True
    )

    word_documents = word_vectorizer.fit_transform(documents)
    word_queries = word_vectorizer.transform(queries)
    char_documents = char_vectorizer.fit_transform(documents)
    char_queries = char_vectorizer.transform(queries)

    n_docs, avgdl, doc_freq = bm25_statistics(documents)
    hr_set = set(normalize_text(" ".join(HR_TERMS)).split())
    intent_set = set(normalize_text(" ".join(INTENT_TERMS)).split())

    rows = []
    for query_id, query in enumerate(queries):
        query_set = set(query.split())
        query_hr = query_set & hr_set
        query_intent = query_set & intent_set

        bm25_scores = bm25_score_documents(
            documents, query, n_docs, avgdl, doc_freq
        )
        word_scores = cosine_similarity(
            word_documents, word_queries[query_id]
        ).ravel()
        char_scores = cosine_similarity(
            char_documents, char_queries[query_id]
        ).ravel()

        for i, row in profile_groups.reset_index(drop=True).iterrows():
            doc_set = set(documents[i].split())
            union = query_set | doc_set
            rows.append({
                "query_id": query_id,
                "query_text": QUERIES[query_id],
                "group_key": row["group_key"],
                "canonical_id": int(row["canonical_id"]),
                "job_title": row["job_title"],
                "location": row["location"],
                "duplicate_count": int(row["duplicate_count"]),
                "relevance": float(row["relevance"]),
                "bm25": float(bm25_scores[i]),
                "tfidf_word": float(word_scores[i]),
                "tfidf_char": float(char_scores[i]),
                "jaccard": len(query_set & doc_set) / len(union) if union else 0.0,
                "containment": len(query_set & doc_set) / len(query_set) if query_set else 0.0,
                "hr_overlap": len(query_hr & doc_set) / len(query_hr) if query_hr else 0.0,
                "intent_overlap": (
                    len(query_intent & doc_set) / len(query_intent)
                    if query_intent else 0.0
                ),
            })
    return pd.DataFrame(rows)

features = build_full_features(groups)
assert set(SELECTED_FEATURES).issubset(features.columns)
display(features.head())

stage1_model = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=SELECTED_ALPHA)),
])
stage1_model.fit(features[SELECTED_FEATURES], features["relevance"])

features["prediction"] = stage1_model.predict(features[SELECTED_FEATURES])
features["prediction_norm"] = minmax(features["prediction"])

stage1_rank = (
    features.groupby(
        ["group_key", "canonical_id", "job_title", "location",
         "duplicate_count", "relevance"],
        as_index=False
    )
    .agg(stage1_score=("prediction_norm", "max"))
    .sort_values(["stage1_score", "canonical_id"], ascending=[False, True])
    .reset_index(drop=True)
)
stage1_rank["stage1_rank"] = np.arange(1, len(stage1_rank) + 1)

display(
    stage1_rank[
        ["stage1_rank", "canonical_id", "job_title", "relevance", "stage1_score"]
    ].head(20)
)

top = stage1_rank.head(20).sort_values("stage1_score")
plt.figure(figsize=(8, 6))
plt.barh(top["job_title"].str.slice(0, 48), top["stage1_score"])
plt.xlabel("Stage-1 score")
plt.title("Top 20 Stage-1 candidate profiles")
plt.tight_layout()
plt.show()

def rocchio_scores(
    profile_groups,
    starred_ids=(3,),
    rejected_ids=(),
    alpha=1.0,
    beta=0.75,
    gamma=0.15,
):
    id_to_group = {}
    for _, row in profile_groups.iterrows():
        for source_id in row["member_ids"]:
            id_to_group[int(source_id)] = row["group_key"]

    starred_groups = {id_to_group[int(i)] for i in starred_ids}
    rejected_groups = {id_to_group[int(i)] for i in rejected_ids}

    documents = profile_groups["title_norm"].tolist()
    queries = [normalize_text(q) for q in QUERIES]

    vectorizer = TfidfVectorizer(
        tokenizer=str.split, preprocessor=None, token_pattern=None,
        ngram_range=(1, 2), sublinear_tf=True
    )
    document_matrix = vectorizer.fit_transform(documents)
    query_matrix = vectorizer.transform(queries)

    group_to_row = {
        group_key: row_id
        for row_id, group_key in enumerate(profile_groups["group_key"])
    }

    def centroid(group_keys):
        row_ids = [group_to_row[key] for key in group_keys]
        if not row_ids:
            return np.zeros((1, document_matrix.shape[1]))
        return np.asarray(document_matrix[row_ids].mean(axis=0))

    positive_centroid = centroid(starred_groups)
    negative_centroid = centroid(rejected_groups)

    query_score_vectors = []
    for query_id in range(query_matrix.shape[0]):
        adjusted_query = (
            alpha * query_matrix[query_id].toarray()
            + beta * positive_centroid
            - gamma * negative_centroid
        )
        query_score_vectors.append(
            cosine_similarity(document_matrix, adjusted_query).ravel()
        )

    raw_feedback = np.max(np.vstack(query_score_vectors), axis=0)

    feedback = profile_groups[
        ["group_key", "canonical_id", "job_title"]
    ].copy()
    feedback["feedback_score"] = minmax(raw_feedback)
    feedback["starred"] = feedback["group_key"].isin(starred_groups)
    feedback["rejected"] = feedback["group_key"].isin(rejected_groups)
    return feedback

feedback = rocchio_scores(
    groups,
    starred_ids=(3,),
    rejected_ids=(),
    alpha=1.0,
    beta=0.75,
    gamma=0.15,
)

stage2_rank = stage1_rank.merge(
    feedback[["group_key", "feedback_score", "starred", "rejected"]],
    on="group_key",
    how="left",
)
stage2_rank["stage2_score"] = (
    0.65 * stage2_rank["stage1_score"]
    + 0.35 * stage2_rank["feedback_score"]
)
stage2_rank = (
    stage2_rank.sort_values(
        ["stage2_score", "canonical_id"], ascending=[False, True]
    )
    .reset_index(drop=True)
)
stage2_rank["stage2_rank"] = np.arange(1, len(stage2_rank) + 1)
stage2_rank["rank_change"] = (
    stage2_rank["stage1_rank"] - stage2_rank["stage2_rank"]
)

display(
    stage2_rank[
        ["stage2_rank", "canonical_id", "job_title",
         "stage1_rank", "rank_change",
         "stage1_score", "feedback_score", "stage2_score", "starred"]
    ].head(20)
)

largest_movements = (
    stage2_rank.assign(abs_rank_change=stage2_rank["rank_change"].abs())
    .sort_values(
        ["abs_rank_change", "stage2_rank"],
        ascending=[False, True]
    )
    [["canonical_id", "job_title", "stage1_rank", "stage2_rank", "rank_change"]]
    .head(15)
)
display(largest_movements)

print("Stage-1 top title:", stage1_rank.iloc[0]["job_title"])
print("Stage-1 top canonical ID:", int(stage1_rank.iloc[0]["canonical_id"]))
print("Stage-2 top title:", stage2_rank.iloc[0]["job_title"])
print("Stage-2 top canonical ID:", int(stage2_rank.iloc[0]["canonical_id"]))
print("Demonstration starred source ID: 3")
print("Stage-2 blend: 65% Stage-1 + 35% feedback")

checkpoint = {
    "selected_alpha": float(SELECTED_ALPHA),
    "selected_features": list(SELECTED_FEATURES),
    "stage1_top_id": int(stage1_rank.iloc[0]["canonical_id"]),
    "stage1_top_title": str(stage1_rank.iloc[0]["job_title"]),
    "stage2_top_id": int(stage2_rank.iloc[0]["canonical_id"]),
    "stage2_top_title": str(stage2_rank.iloc[0]["job_title"]),
}
print("CHECKPOINT", checkpoint)

CHECKPOINT {'selected_alpha': 10.0, 'selected_features': ['bm25', 'tfidf_word', 'tfidf_char', 'jaccard', 'containment', 'hr_overlap', 'intent_overlap'], 'stage1_top_id': 99, 'stage1_top_title': 'Seeking Human Resources Position', 'stage2_top_id': 3, 'stage2_top_title': 'Aspiring Human Resources Professional'}
